# Colab GPU Launcher

Notebook này chạy project `NLP-test` trên Google Colab GPU.

Ý tưởng quan trọng: **không train trực tiếp trong Google Drive** vì Drive mount có thể bị lỗi `Transport endpoint is not connected`. Launcher sẽ copy project từ Drive vào `/content/NLP-test`, chạy ở local runtime, rồi sync `reports/` và `models/` ngược về Drive.

Trước khi chạy:

1. Upload/copy thư mục `NLP-test` vào Google Drive, ví dụ `MyDrive/NLP-test`.
2. Trong Colab chọn `Runtime > Change runtime type > GPU`.
3. Chạy các cell bên dưới theo thứ tự.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Bạn chưa bật GPU: Runtime > Change runtime type > GPU')

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Copy project from Drive to local Colab runtime

Nếu project của bạn nằm vị trí khác, sửa `DRIVE_PROJECT_DIR`.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/NLP-test')
LOCAL_PROJECT_DIR = Path('/content/NLP-test')
assert (DRIVE_PROJECT_DIR / 'src').exists(), f'Không thấy project tại {DRIVE_PROJECT_DIR}. Hãy sửa DRIVE_PROJECT_DIR cho đúng.'
print('Drive project:', DRIVE_PROJECT_DIR)
print('Local project:', LOCAL_PROJECT_DIR)

In [ ]:
# Copy source/data/notebooks to local runtime. Running locally avoids Google Drive mount instability.
!rm -rf /content/NLP-test
!mkdir -p /content/NLP-test
!rsync -a --info=progress2   --exclude='models/deep_ner/'   --exclude='reports/deep_ner/'   "$DRIVE_PROJECT_DIR/" "$LOCAL_PROJECT_DIR/"
%cd /content/NLP-test
!find data -maxdepth 3 -type f -print | sort | sed -n '1,80p'

from pathlib import Path
required_raw = [
    Path('data/raw/invoices.csv.zip'),
    Path('data/raw/SROIE_datasetv2.zip'),
]
missing = [str(path) for path in required_raw if not path.exists()]
assert not missing, 'Missing required raw dataset file(s): ' + ', '.join(missing) + '. Upload them to MyDrive/NLP-test/data/raw/ first.'

## Install dependencies

In [ ]:
%pip install -q -r requirements-deep.txt

## Prepare data

Dùng `data/raw/invoices.csv.zip` làm dataset A để không phụ thuộc vào `data/interim/invoices.csv`.

In [ ]:
!python src/prepare_ner_data.py   --dataset-a data/raw/invoices.csv.zip   --dataset-b data/raw/SROIE_datasetv2.zip

## Run experiments

Khuyến nghị chạy từng cell một để theo dõi thời gian và VRAM.

In [ ]:
# Token Classification models
!cd /content/NLP-test && jupyter nbconvert --to notebook --execute notebooks/01_transformer_token_classification_khoi.ipynb --output 01_transformer_token_classification_khoi.executed.ipynb --ExecutePreprocessor.timeout=-1

In [ ]:
# CRF models
!cd /content/NLP-test && jupyter nbconvert --to notebook --execute notebooks/02_transformer_crf_models.ipynb --output 02_transformer_crf_models.executed.ipynb --ExecutePreprocessor.timeout=-1

In [ ]:
# Global Pointer + Global Context
!cd /content/NLP-test && jupyter nbconvert --to notebook --execute notebooks/03_global_pointer_and_global_context.ipynb --output 03_global_pointer_and_global_context.executed.ipynb --ExecutePreprocessor.timeout=-1

In [ ]:
# Select best model
!cd /content/NLP-test && jupyter nbconvert --to notebook --execute notebooks/04_select_best_model.ipynb --output 04_select_best_model.executed.ipynb --ExecutePreprocessor.timeout=-1
!cat /content/NLP-test/reports/deep_ner/leaderboard.md

## Sync results back to Google Drive

In [ ]:
!mkdir -p "$DRIVE_PROJECT_DIR/reports" "$DRIVE_PROJECT_DIR/models"
!rsync -a /content/NLP-test/reports/deep_ner "$DRIVE_PROJECT_DIR/reports/"
!rsync -a /content/NLP-test/models/deep_ner "$DRIVE_PROJECT_DIR/models/"
!rsync -a /content/NLP-test/notebooks/*.executed.ipynb "$DRIVE_PROJECT_DIR/notebooks/"
print('Synced reports/models/executed notebooks back to Drive.')